# Baseline CNN, trained on Kaggle

ITRI626 group project. This notebook trains the from-scratch baseline CNN on the shared
70/15/15 split of Stanford Dogs. The code lives in the group repository and is cloned
below, so nothing important is written inside notebook cells.

## Before running anything, set these three things up

1. **GPU on.** Right-hand panel, Session options, Accelerator, choose **GPU T4 x2** or
   **GPU P100**. Without this the run takes days rather than an hour.
2. **Internet on.** Same panel. The clone in step 2 needs it.
3. **Add the dataset.** Top right, **Add Input**, search `stanford-dogs-dataset` and add
   *Stanford Dogs Dataset* by Jessica Li. It then appears under `/kaggle/input/`.

Kaggle gives about 30 GPU hours a week and stops idle sessions, so for the long run use
**Save Version, Run All** and let it finish without you watching.

## Step 1: check the machine

Confirms a GPU is actually attached. If `cuda available` prints `False`, stop and turn the
accelerator on, otherwise everything below runs on the CPU and will be unusably slow.

In [ ]:
import torch

print("torch          ", torch.__version__)
print("cuda available ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu            ", torch.cuda.get_device_name(0))
    print("memory gb      ", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

!ls /kaggle/input

## Step 2: fetch only my folder from the repository

The repository contains all 20,580 photographs, roughly 800 MB, which there is no reason
to download here because Kaggle already hosts the dataset. A **sparse checkout** takes the
`Lindani_Model` folder and nothing else, so this finishes in seconds.

If the sparse checkout fails for any reason, fall back to a plain clone. It works, but it
downloads the photographs as well and takes a few minutes:

    !rm -rf {WORK} && git clone --depth 1 --branch {BRANCH} {REPO} {WORK}

In [ ]:
REPO = "https://github.com/JP-NWU-honours-2026/Dog_Classifier.git"
BRANCH = "Lindani-branch"
WORK = "/kaggle/working/repo"

!rm -rf {WORK}
!git clone --filter=blob:none --no-checkout --depth 1 --branch {BRANCH} {REPO} {WORK}
!cd {WORK} && git sparse-checkout init --cone && git sparse-checkout set Lindani_Model && git checkout
!ls {WORK}/Lindani_Model

## Step 3: point the code at Kaggle's copy of the photographs

`dataset.py` looks for the breed folders in several places and uses the first one it finds.
`DOG_IMAGES_DIR` is checked first, so setting it here is all that is needed. No code
changes, which is the point of building it that way.

If the path below does not exist, look at the `ls /kaggle/input` output from step 1 and
correct it.

In [ ]:
import os
from pathlib import Path

candidates = [
    "/kaggle/input/stanford-dogs-dataset/images/Images",
    "/kaggle/input/stanford-dogs-dataset/Images",
]
images_dir = next((c for c in candidates if Path(c).is_dir()), None)
assert images_dir, f"none of these exist: {candidates}"

os.environ["DOG_IMAGES_DIR"] = images_dir
print("using", images_dir)
print("breed folders:", len(list(Path(images_dir).iterdir())))

## Step 4: check the split and the photographs agree

Runs the self-test in `dataset.py`. It must report 14,355 training, 3,025 validation and
3,200 test photographs, 120 breeds, and pass its label check. If it cannot find the
photographs it will say so plainly.

In [ ]:
!cd {WORK} && python Lindani_Model/dataset.py

## Step 5: smoke test, two epochs over a handful of batches

Proves the whole path works on the GPU before committing an hour to it. Takes a minute or
two.

**Check three things in the output:**

- `Training on cuda` followed by the GPU name, not `cpu`
- `1,631,128 trainable parameters`
- the loss falls between the two epochs

Accuracy here is meaningless. Eight batches teaches the model almost nothing.

In [ ]:
!cd {WORK} && python Lindani_Model/train.py --quick --name kaggle_smoke --num-workers 2 --overwrite

## Step 6: the real run

Sixty epochs with early stopping on validation macro F1, patience 5, exactly as recorded
in `MODEL_DESIGN.md`. Expect roughly one to two minutes per epoch, so one to two hours in
total, less if early stopping triggers.

Everything needed for the report is written automatically to
`Lindani_Model/results/runs/baseline_cnn`: `run_config.json`, `metrics.csv`,
`best_model.pth` and the validation confusion matrix of the best epoch.

**The test set is not touched by this script.** It is scored once, later, by `evaluate.py`.

One group decision to settle before the final run: whether all three models use label
smoothing. If the group agrees on 0.1, add `--label-smoothing 0.1` below.

In [ ]:
!cd {WORK} && python Lindani_Model/train.py --epochs 60 --batch-size 64 --num-workers 2 --name baseline_cnn --overwrite

## Step 7: look at the run before leaving

Prints the last few epochs and the headline figures. Two things to note for the report:
whether **early stopping** triggered or the **epoch cap** was reached, and how far the
training and validation curves diverged, which is how overfitting shows up.

In [ ]:
import json

run = Path(WORK) / "Lindani_Model/results/runs/baseline_cnn"
print(open(run / "metrics.csv").read().strip().split("\n")[0])
for line in open(run / "metrics.csv").read().strip().split("\n")[-6:]:
    print(line)

config = json.loads((run / "run_config.json").read_text())
for key in ["epochs_planned", "epochs_run", "stopped_early", "epoch_cap_reached",
            "best_epoch", "best_val_macro_f1", "total_minutes"]:
    print(f"{key:<22} {config.get(key)}")
print(f"{'gpu':<22} {config['environment'].get('gpu')}")

## Step 8: take the results off Kaggle

Kaggle wipes the session when it ends. Only `/kaggle/working` survives, and only if the
notebook version is saved, so copy the run folder somewhere obvious and download it.

**What to do with each file once it is on your laptop**, in
`Lindani_Model/results/runs/baseline_cnn/`:

| File | What to do |
|---|---|
| `run_config.json` | Commit. This is the reproducibility record |
| `metrics.csv` | Commit. The training and validation curves come from this |
| `val_confusion_best.csv` | Commit. Small, and useful for error analysis |
| `best_model.pth` | **Do not commit.** Weights are ignored by git on purpose. Keep it for `evaluate.py` |

In [ ]:
import shutil

destination = Path("/kaggle/working/baseline_cnn_run")
shutil.rmtree(destination, ignore_errors=True)
shutil.copytree(run, destination)

for path in sorted(destination.iterdir()):
    print(f"{path.name:<26} {path.stat().st_size / 1024:>9.0f} kB")
print("\nDownload these from the Output tab on the right.")